# StreamLoop — Tuning the Churn Model

We prioritize **recall**: a false negative means a customer likely to leave is missed, while a false positive generally means an unnecessary retention offer. Precision and F1 provide additional context. The held-out test set is reserved for exactly one baseline evaluation and one final tuned-model evaluation; all preprocessing is fit inside the model pipeline.

## 1. Load and minimally clean the source data

Download the IBM Telco Customer Churn CSV directly. Blank `TotalCharges` entries are converted to missing values and imputed later, inside the training-fitted preprocessing pipeline. `customerID` is removed and the target is encoded as 1/0.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import randint

DATA_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_URL)
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(r'^\s*$', np.nan, regex=True), errors='coerce')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype('int8')
X = df.drop(columns='Churn')
y = df['Churn']
print(f'Dataset: {X.shape[0]:,} rows × {X.shape[1]} predictors; churn rate: {y.mean():.3f}')
print(f'Missing TotalCharges values to be imputed: {X["TotalCharges"].isna().sum()}')
display(df.head())

Dataset: 7,043 rows × 19 predictors; churn rate: 0.265
Missing TotalCharges values to be imputed: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 2. Split first, then define all preprocessing in a pipeline

The stratified split is performed before any transformer is fitted. Numeric columns use median imputation. Every other predictor is treated as categorical, with most-frequent imputation and one-hot encoding that safely ignores categories not seen in training.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numeric_features]
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', SimpleImputer(strategy='median'), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

def make_pipeline():
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42)),
    ])

print(f'Train: {X_train.shape}; test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.3f}; test churn rate: {y_test.mean():.3f}')

Train: (5634, 19); test: (1409, 19)
Train churn rate: 0.265; test churn rate: 0.265


## 3. Default Random Forest baseline

Fit the default Random Forest pipeline on training data. The following is the baseline's **single** test-set evaluation; do not reuse test metrics for model selection.

In [3]:
baseline_model = make_pipeline()
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)
baseline_metrics = {
    'recall': recall_score(y_test, baseline_predictions),
    'precision': precision_score(y_test, baseline_predictions, zero_division=0),
    'f1': f1_score(y_test, baseline_predictions, zero_division=0),
}
print('Baseline test metrics (one evaluation):')
display(pd.DataFrame([baseline_metrics], index=['Default Random Forest']))

Baseline test metrics (one evaluation):


,recall,precision,f1
Default Random Forest,0.475936,0.615917,0.536953


## 4. Broad randomized search — training data only

Five-fold cross-validation optimizes recall on `X_train`/`y_train` only. The broad distribution explores tree count, depth, split/leaf thresholds, feature subsampling, and class weighting. No test data is passed to either search.

In [4]:
random_param_distributions = {
    'classifier__n_estimators': randint(100, 501),
    'classifier__max_depth': [None, 5, 8, 12, 16, 20, 30],
    'classifier__min_samples_split': randint(2, 21),
    'classifier__min_samples_leaf': randint(1, 11),
    'classifier__max_features': ['sqrt', 'log2', 0.5, 0.8],
    'classifier__class_weight': [None, 'balanced', 'balanced_subsample'],
}
random_search = RandomizedSearchCV(
    estimator=make_pipeline(),
    param_distributions=random_param_distributions,
    n_iter=24, cv=5, scoring='recall', n_jobs=-1,
    random_state=42, refit=True, return_train_score=False,
)
random_search.fit(X_train, y_train)
random_results = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score')
print(f'Randomized search candidates: {len(random_results)}; best mean CV recall: {random_search.best_score_:.4f}')
display(random_results[['mean_test_score', 'std_test_score', 'rank_test_score', 'params']].head(10))
print('Best randomized parameters:')
print(random_search.best_params_)

Randomized search candidates: 24; best mean CV recall: 0.7906


,mean_test_score,std_test_score,rank_test_score,params
5,0.790635,0.021737,1,{'classifier__class_weight': 'balanced_subsamp...
15,0.777926,0.009600,2,"{'classifier__class_weight': 'balanced', 'clas..."
11,0.775920,0.007015,3,"{'classifier__class_weight': 'balanced', 'clas..."
18,0.774582,0.007204,4,{'classifier__class_weight': 'balanced_subsamp...
8,0.769231,0.011586,5,"{'classifier__class_weight': 'balanced', 'clas..."
21,0.767893,0.012970,6,"{'classifier__class_weight': 'balanced', 'clas..."
4,0.759197,0.005983,7,"{'classifier__class_weight': 'balanced', 'clas..."
0,0.758528,0.009317,8,{'classifier__class_weight': 'balanced_subsamp...
3,0.754515,0.005434,9,"{'classifier__class_weight': 'balanced', 'clas..."
13,0.754515,0.008082,10,"{'classifier__class_weight': 'balanced', 'clas..."


Best randomized parameters:
{'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 5, 'classifier__max_features': 0.8, 'classifier__min_samples_leaf': 3, 'classifier__min_samples_split': 13, 'classifier__n_estimators': 154}


## 5. Focused grid search — training data only

Inspect the randomized results to define a compact neighborhood around strong settings. The candidate list is derived only from training-fold CV results and the randomized best parameters. Grid search again uses five-fold recall and refits its best pipeline on all training data.

In [5]:
# Summarize strong randomized candidates and define a narrow grid around the CV winner.
strong_random = random_results.head(5)
best_random = random_search.best_params_
def neighborhood(value, options):
    return sorted(set([value] + [candidate for candidate in options if candidate != value]), key=lambda x: (x is None, str(x)))

depth_options = [best_random['classifier__max_depth']] + [d for d in [None, 8, 12, 16, 20] if d != best_random['classifier__max_depth']]
leaf = best_random['classifier__min_samples_leaf']
split = best_random['classifier__min_samples_split']
estimators = best_random['classifier__n_estimators']
features = best_random['classifier__max_features']
weights = best_random['classifier__class_weight']

grid_param_grid = {
    'classifier__n_estimators': sorted(set([max(100, estimators - 100), estimators, min(500, estimators + 100)])),
    'classifier__max_depth': depth_options[:3],
    'classifier__min_samples_leaf': sorted(set([max(1, leaf - 1), leaf, leaf + 1])),
    'classifier__min_samples_split': sorted(set([max(2, split - 2), split, split + 2])),
    'classifier__max_features': [features],
    'classifier__class_weight': [weights],
}
print('Top randomized CV candidates (training folds only):')
display(strong_random[['mean_test_score', 'std_test_score', 'params']])
print('Focused grid:')
print(grid_param_grid)

Top randomized CV candidates (training folds only):


,mean_test_score,std_test_score,params
5,0.790635,0.021737,{'classifier__class_weight': 'balanced_subsamp...
15,0.777926,0.009600,"{'classifier__class_weight': 'balanced', 'clas..."
11,0.775920,0.007015,"{'classifier__class_weight': 'balanced', 'clas..."
18,0.774582,0.007204,{'classifier__class_weight': 'balanced_subsamp...
8,0.769231,0.011586,"{'classifier__class_weight': 'balanced', 'clas..."


Focused grid:
{'classifier__n_estimators': [100, 154, 254], 'classifier__max_depth': [5, None, 8], 'classifier__min_samples_leaf': [2, 3, 4], 'classifier__min_samples_split': [11, 13, 15], 'classifier__max_features': [0.8], 'classifier__class_weight': ['balanced_subsample']}


In [6]:
grid_search = GridSearchCV(
    estimator=make_pipeline(), param_grid=grid_param_grid,
    cv=5, scoring='recall', n_jobs=-1, refit=True, return_train_score=False,
)
grid_search.fit(X_train, y_train)
grid_results = pd.DataFrame(grid_search.cv_results_).sort_values(
    ['mean_test_score', 'std_test_score'], ascending=[False, True]
).reset_index(drop=True)
top_grid_candidates = grid_results.head(5)
print(f'Grid candidates: {len(grid_results)}; best mean CV recall: {grid_search.best_score_:.4f}')
display(top_grid_candidates[['mean_test_score', 'std_test_score', 'rank_test_score', 'params']])
print('Grid-search best parameters:')
print(grid_search.best_params_)

Grid candidates: 81; best mean CV recall: 0.7926


,mean_test_score,std_test_score,rank_test_score,params
0,0.792642,0.019502,1,{'classifier__class_weight': 'balanced_subsamp...
1,0.792642,0.020399,1,{'classifier__class_weight': 'balanced_subsamp...
2,0.792642,0.020940,1,{'classifier__class_weight': 'balanced_subsamp...
3,0.791973,0.020442,4,{'classifier__class_weight': 'balanced_subsamp...
4,0.791973,0.020442,4,{'classifier__class_weight': 'balanced_subsamp...


Grid-search best parameters:
{'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 5, 'classifier__max_features': 0.8, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 11, 'classifier__n_estimators': 100}


## 6. Final model evaluation

Use `grid_search.best_estimator_` exactly as refitted by `GridSearchCV(refit=True)`; do not manually refit. This is the tuned model's **single** test-set evaluation. The top-five CV table reports both mean and standard deviation: a near-best mean with lower fold-to-fold variation can be a more dependable choice than a tiny mean-score advantage with unstable recall.

In [7]:
final_model = grid_search.best_estimator_
tuned_predictions = final_model.predict(X_test)
tuned_metrics = {
    'recall': recall_score(y_test, tuned_predictions),
    'precision': precision_score(y_test, tuned_predictions, zero_division=0),
    'f1': f1_score(y_test, tuned_predictions, zero_division=0),
}
comparison = pd.DataFrame([baseline_metrics, tuned_metrics], index=['Baseline', 'Tuned (GridSearchCV)'])
comparison['recall_change_vs_baseline'] = [0.0, tuned_metrics['recall'] - baseline_metrics['recall']]
print('Final test-set comparison (baseline and final model each evaluated once):')
display(comparison)
print('Final hyperparameters:')
print(grid_search.best_params_)
print('Top five grid CV candidates:')
display(top_grid_candidates[['mean_test_score', 'std_test_score', 'params']])

Final test-set comparison (baseline and final model each evaluated once):


,recall,precision,f1,recall_change_vs_baseline
Baseline,0.475936,0.615917,0.536953,0.000000
Tuned (GridSearchCV),0.775401,0.532110,0.631121,0.299465


Final hyperparameters:
{'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 5, 'classifier__max_features': 0.8, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 11, 'classifier__n_estimators': 100}
Top five grid CV candidates:


,mean_test_score,std_test_score,params
0,0.792642,0.019502,{'classifier__class_weight': 'balanced_subsamp...
1,0.792642,0.020399,{'classifier__class_weight': 'balanced_subsamp...
2,0.792642,0.020940,{'classifier__class_weight': 'balanced_subsamp...
3,0.791973,0.020442,{'classifier__class_weight': 'balanced_subsamp...
4,0.791973,0.020442,{'classifier__class_weight': 'balanced_subsamp...


## Interpretation

Recall remains the selection metric because missed churners have the greater stated business cost. Compare the top candidates' `std_test_score` alongside their means: lower dispersion across folds suggests more stable recall. Precision and F1 show the corresponding retention-offer trade-off. The held-out test comparison is presented only after all tuning choices are complete.